Load the source PDFs that seed the retrieval pipeline.


In [1]:
from typing import List, TypedDict
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()


/tmp/ipykernel_20187/425241269.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/amir/LangGraph/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Load the source PDFs that seed the retrieval pipeline.


In [2]:
docs = (
    PyPDFLoader("../books/book1.pdf").load() +
    PyPDFLoader("../books/book2.pdf").load() +
    PyPDFLoader("../books/book3.pdf").load()
)


Sanity-check the loaded corpus before chunking and indexing.


In [3]:
len(docs)

2123

Split the documents into overlap-aware chunks and normalize the text payloads.


In [4]:
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)

for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")


Build the embedding index and expose it through a retriever for graph nodes.


In [5]:
embeddings = HuggingFaceBgeEmbeddings("sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)


/tmp/ipykernel_20187/2385454757.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings("sentence-transformers/all-MiniLM-L6-v2")


TypeError: HuggingFaceBgeEmbeddings.__init__() takes 1 positional argument but 2 were given

Build the embedding index and expose it through a retriever for graph nodes.


In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})


Instantiate the chat model that drives routing, evaluation, or generation.


In [ ]:
llm = ChatGroq(model_name="llama-3.3-70b-versatile")


Define the typed shared state contract passed between workflow nodes.


In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str


Implement the retrieval step that fetches relevant context for the active question.


In [ ]:
def retrieve(state):
    q = state["question"]
    return {"docs": retriever.invoke(q)}


Define the prompt and response synthesis logic for the current graph stage.


In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": out.content}


Wire the nodes and routing rules into an executable LangGraph workflow.


In [ ]:
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app


Run a representative query through the compiled graph to validate behavior.


In [ ]:
res = app.invoke({"question": "WHat is a vision transformer in deep learning.", "docs": [], "answer": ""})
print(res["answer"])


Inspect retrieved context to verify what the retriever surfaced.


In [ ]:
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)
